In [131]:
import numpy as np
from activators import ReluActivator,IdentityActivator

如果要训练真正的RNN网络，还需要：
实现输出层（如全连接层）
实现损失函数（如MSE、交叉熵）
在输出层中计算 sensitivity_array 并传给RNN层

In [132]:
def element_wise_op(array, op):
    for i in np.nditer(array,
                       op_flags=['readwrite']):
        i[...] = op(i)

In [133]:
from functools import reduce


class RecurrentLayer(object):
    def __init__(self,input_width,state_width,activator,learning_rate):
        '''
        :param input_width:
        :param state_width:隐藏层神经元数量
        :param activator:
        :param learning_rate:
        '''
        self.input_width = input_width
        self.state_width = state_width
        self.activator = activator
        self.learning_rate = learning_rate

        self.times=0 #当前处理到第几个时间步
        self.state_list=[] #保存每个时刻的状态
        # 初始化状态s0
        self.state_list.append(np.zeros((state_width,1)))
        # 初始化输入权重矩阵U
        self.U=np.random.uniform(-1e-4,1e-4,(state_width,input_width))
        # 初始化状态权重矩阵W
        self.W=np.random.uniform(-1e-4,1e-4,(state_width,state_width))
    def forward(self,input_array):
        '''
        :param input_array:
        :return: 增加时间步、计算s、应用激活函数f、保存到state_list
        '''
        self.times+=1
        state=np.dot(self.U,input_array)+np.dot(self.W,self.state_list[-1])
        element_wise_op(state,self.activator.forward)
        self.state_list.append(state)
    def backward(self,sensitivity_array,activator):
        self.calc_delta(sensitivity_array,activator)
        self.calc_gradient()
    def update(self):
        self.W-=self.learning_rate * self.gradient
    def calc_delta(self,sensitivity_array,activator):
        '''
        对于最后一个时刻 T：
            δ_T = (∂E/∂s_T) ⊙ f'(s_T)

        对于前面的时刻 t（从 T-1 到 1）：
            δ_t = (W^T · δ_{t+1}) ⊙ f'(s_t)
        '''
        self.delta_list=[]
        for i in range(self.times):
            self.delta_list.append(np.zeros((self.state_width,1)))
        self.delta_list.append(sensitivity_array)

        for k in range(self.times-1,0,-1):
            self.calc_delta_k(k,activator)
    def calc_delta_k(self,k,activator):
        '''
        根据 k+1 时刻的误差项计算 k 时刻的误差项
        '''
        state = self.state_list[k+1].copy()
        f_prime = state.copy()
        element_wise_op(f_prime, activator.backward)
        self.delta_list[k] = np.dot(self.W.T, self.delta_list[k+1]) * f_prime
    def calc_gradient(self):
        self.gradient_list=[]
        for t in range(self.times+1):
            self.gradient_list.append(np.zeros((self.state_width,self.state_width)))
        for t in range(self.times,0,-1):
            self.calc_gradient_t(t)
        self.gradient=reduce(lambda a,b:a+b,self.gradient_list,self.gradient_list[0])
    def calc_gradient_t(self,t):
        gradient= np.dot(self.delta_list[t],self.state_list[t-1].T)
        self.gradient_list[t] = gradient

    def reset_state(self):
        self.times=0
        self.state_list=[]
        self.state_list.append(np.zeros((self.state_width,1)))






In [134]:
def data_set():
    x = [np.array([[1], [2], [3]]),
         np.array([[2], [3], [4]])]
    d = np.array([[1], [2]])
    return x, d


def gradient_check():
    '''
    梯度检查
    '''
    # 设计一个误差函数，取所有节点输出项之和
    error_function = lambda o: o.sum()

    rl = RecurrentLayer(3, 2, IdentityActivator(), 1e-3)

    # 计算forward值
    x, d = data_set()
    rl.forward(x[0])
    rl.forward(x[1])

    # 求取sensitivity map
    sensitivity_array = np.ones(rl.state_list[-1].shape,
                                dtype=np.float64)
    # 计算梯度
    rl.backward(sensitivity_array, IdentityActivator())

    # 检查梯度
    epsilon = 10e-4
    for i in range(rl.W.shape[0]):
        for j in range(rl.W.shape[1]):
            rl.W[i,j] += epsilon
            rl.reset_state()
            rl.forward(x[0])
            rl.forward(x[1])
            err1 = error_function(rl.state_list[-1])
            rl.W[i,j] -= 2*epsilon
            rl.reset_state()
            rl.forward(x[0])
            rl.forward(x[1])
            err2 = error_function(rl.state_list[-1])
            expect_grad = (err1 - err2) / (2 * epsilon)
            rl.W[i,j] += epsilon
            print('weights(%d,%d): expected - actual %f - %f' % (
                i, j, expect_grad, rl.gradient[i, j]))


def test():
    l = RecurrentLayer(3, 2, ReluActivator(), 1e-3)
    x, d = data_set()
    l.forward(x[0])
    l.forward(x[1])
    l.backward(d, ReluActivator())
    return l



In [136]:
if __name__ == '__main__':
    rl = test()
    print("训练后的权重矩阵 W:")
    print(rl.W)
    print("\n训练后的输入权重矩阵 U:")
    print(rl.U)
    print("\n各时刻的状态:")
    for i, state in enumerate(rl.state_list):
        formatted = [f"{val:.10f}" for val in state.flatten()]
        print(f"s_{i}: [{', '.join(formatted)}]")

训练后的权重矩阵 W:
[[ 4.69849030e-05  4.71410711e-05]
 [-7.69724295e-05 -8.28191990e-05]]

训练后的输入权重矩阵 U:
[[ 7.07951868e-05  5.23181473e-05 -4.04980934e-05]
 [ 2.45401775e-05  6.50316217e-05 -2.78487596e-05]]

各时刻的状态:
s_0: [0.0000000000, 0.0000000000]
s_1: [0.0000539372, 0.0000710571]
s_2: [0.0001365583, 0.0001327701]


In [140]:
class RecurrentLayer(object):
    def __init__(self, input_width, state_width, activator, learning_rate):
        self.input_width = input_width
        self.state_width = state_width
        self.activator = activator
        self.learning_rate = learning_rate
        self.times = 0
        self.state_list = []
        self.input_list = []  # 保存每个时刻的输入
        self.state_list.append(np.zeros((state_width, 1)))

        init_range = 0.01
        self.U = np.random.uniform(-init_range, init_range,
            (state_width, input_width))
        self.W = np.random.uniform(-init_range, init_range,
            (state_width, state_width))

    def forward(self, input_array):
        '''前向传播'''
        self.input_list.append(input_array.copy())  # 保存输入
        self.times += 1
        state = np.dot(self.U, input_array) + np.dot(self.W, self.state_list[-1])
        element_wise_op(state, self.activator.forward)
        self.state_list.append(state)

    def backward(self, sensitivity_array, activator):
        '''反向传播'''
        self.calc_delta(sensitivity_array, activator)
        self.calc_gradient()

    def update(self):
        '''更新权重'''
        self.U -= self.learning_rate * self.U_grad
        self.W -= self.learning_rate * self.W_grad

    def calc_delta(self, sensitivity_array, activator):
        '''计算误差项'''
        self.delta_list = []
        for i in range(self.times):
            self.delta_list.append(np.zeros((self.state_width, 1)))
        self.delta_list.append(sensitivity_array)

        for k in range(self.times - 1, 0, -1):
            self.calc_delta_k(k, activator)

    def calc_delta_k(self, k, activator):
        '''计算单个时刻的误差项'''
        state = self.state_list[k+1].copy()
        f_prime = state.copy()
        element_wise_op(f_prime, activator.backward)
        self.delta_list[k] = np.dot(self.W.T, self.delta_list[k+1]) * f_prime

    def calc_gradient(self):
        '''计算 U 和 W 的梯度'''
        self.U_grad = np.zeros_like(self.U)
        self.W_grad = np.zeros_like(self.W)

        # 对每个时刻累加梯度
        for t in range(1, self.times + 1):
            # W 的梯度：∂E/∂W = δ_t · s_{t-1}^T
            self.W_grad += np.dot(self.delta_list[t], self.state_list[t-1].T)

            # U 的梯度：∂E/∂U = δ_t · x_t^T
            if t <= len(self.input_list):
                x_t = self.input_list[t-1]
                self.U_grad += np.dot(self.delta_list[t], x_t.T)

    def reset_state(self):
        '''重置状态'''
        self.times = 0
        self.state_list = []
        self.input_list = []
        self.state_list.append(np.zeros((self.state_width, 1)))


def test():
    print("="*50)
    print("RNN 测试（包含 U 和 W 的更新）")
    print("="*50)

    l = RecurrentLayer(3, 2, ReluActivator(), 0.01)
    x = [np.array([[1], [2], [3]]),
         np.array([[2], [3], [4]])]

    print("\n初始权重:")
    print(f"U 范围: [{l.U.min():.6f}, {l.U.max():.6f}]")
    print(f"W 范围: [{l.W.min():.6f}, {l.W.max():.6f}]")


    print("\n前向传播:")
    l.forward(x[0])
    print(f"  s1: [{l.state_list[1][0,0]:.8f}, {l.state_list[1][1,0]:.8f}]")

    l.forward(x[1])
    print(f"  s2: [{l.state_list[2][0,0]:.8f}, {l.state_list[2][1,0]:.8f}]")


    sensitivity = np.array([[1.0], [1.0]])
    l.backward(sensitivity, ReluActivator())

    print("\n梯度:")
    print(f"  U_grad 范围: [{l.U_grad.min():.6f}, {l.U_grad.max():.6f}]")
    print(f"  W_grad 范围: [{l.W_grad.min():.6f}, {l.W_grad.max():.6f}]")


    l.update()

    print("\n更新后权重:")
    print(f"U 范围: [{l.U.min():.6f}, {l.U.max():.6f}]")
    print(f"W 范围: [{l.W.min():.6f}, {l.W.max():.6f}]")

    l.reset_state()
    l.forward(x[0])
    l.forward(x[1])
    print(f"\n更新后 s2: [{l.state_list[2][0,0]:.8f}, {l.state_list[2][1,0]:.8f}]")

    return l


if __name__ == '__main__':
    test()

RNN 测试（包含 U 和 W 的更新）

初始权重:
U 范围: [-0.004663, 0.008883]
W 范围: [-0.007997, 0.007202]

前向传播:
  s1: [0.00433168, 0.02364466]
  s2: [0.00733786, 0.04017226]

梯度:
  U_grad 范围: [1.996832, 4.042669]
  W_grad 范围: [0.004332, 0.023645]

更新后权重:
U 范围: [-0.040683, -0.011259]
W 范围: [-0.008040, 0.006965]

更新后 s2: [0.00000000, 0.00000000]
